# Image Classification with PyTorch: Airplane, Bird, and Deer

This notebook demonstrates how to use PyTorch to train a model that classifies images into three categories: **airplane**, **bird**, and **deer**. We will use the CIFAR-10 dataset, which contains 10 classes of images, including the three classes we are interested in.

## Table of Contents
1. [Setup](#setup)
2. [Model Definition](#model-definition)
3. [Training](#training)
4. [Evaluation](#evaluation)

---

### 1. Setup <a id="setup"></a>

In [ ]:
path =  '../data-unversioned/p1ch7/'
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn.functional as F
import matplotlib.pyplot as plt
import datetime

In [ ]:
cifar10 = datasets.CIFAR10(path, train =  True, download = True, transform = transforms.Compose([transforms.ToTensor(),
                                                                                                transforms.Normalize((0.4914, 0.4822, 0.4465),
                                                                                                                     (0.2470, 0.2435, 0.2616))]))
cifar10_val = datasets.CIFAR10(path, train = False, download = True, transform = transforms.Compose([transforms.ToTensor(),
                                                                                                transforms.Normalize((0.4914, 0.4822, 0.4465),
                                                                                                                     (0.2470, 0.2435, 0.2616))]))

100%|██████████| 170M/170M [00:04<00:00, 35.3MB/s]


Extracting ../data-unversioned/p1ch7/cifar-10-python.tar.gz to ../data-unversioned/p1ch7/
Files already downloaded and verified


In [ ]:
cifar3 = [(img, label/2) for img, label in cifar10 if label in [0, 2, 4]]
cifar3_val = [(img, label/2)for img, label in cifar10_val if label in [0,2,4]]

In [ ]:
trainloader = torch.utils.data.DataLoader(cifar3, batch_size = 64, shuffle = True)
valLoader = torch.utils.data.DataLoader(cifar3_val, batch_size = 64, shuffle = False)

In [ ]:
img, label = cifar3[10]

In [ ]:
conv = nn.Conv2d(3, 16, kernel_size=3)

In [ ]:
output = conv(img.unsqueeze(0))

#2. Model Definition <a id="model-definition"></a>
Define a Convolutional Neural Network (CNN) model to classify images.

In [ ]:
class Net(nn.Module):
  def __init__(self, n_channel = 32,dropout = None):
    super().__init__()
    self.conv1 = nn.Conv2d(3, n_channel, kernel_size = 3, padding =1)
    self.n_channel = n_channel
    self.dropout = dropout
    if dropout:
      self.conv1_drop =nn.Dropout2d(p = dropout)
    self.conv2 = nn.Conv2d(n_channel, self.n_channel//2, kernel_size = 3, padding =1)
    self.conv3 = nn.conv2d(self.n_channel//2, self.n_channel//2, kernel_size = 3, padding = 1)
    self.fc1 = nn.Linear(4*4*self.n_channel//2, 32)
    self.fc2 = nn.Linear(32, 3)
  def forward(self, x):
    out = torch.max_pool2d(torch.relu(self.conv1(x)), 2)
    if self.dropout:
      out = self.conv1_drop(out)
    out = torch.max_pool2d(torch.relu(self.conv2(out)), 2)
    out1 = out
    out = torch.max_pool2d(torch.relu(self.conv3(out))+out1, 2)
    out = out.view(-1, 4*4*self.n_channel//2)
    out = torch.tanh(self.fc1(out))
    out = self.fc2(out)
    return out

In [ ]:
device = (torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu'))

In [ ]:
model1 = Net().to(device = device)

In [ ]:
import torch.optim as optim
optimizer = optim.SGD(model1.parameters(), lr = 1e-2)

In [ ]:
loss_fn = nn.CrossEntropyLoss()

#3. Training <a id="training"></a>
Train the model using the training dataset.

In [ ]:
def training_loop(n_epochs, model, loss_fn, optimizer, trainloader):
  for epoch in range(n_epochs):
    loss_train = 0.0
    for imgs, labels in trainloader:
      imgs = imgs.to(device = device)
      labels = labels.to(device = device).long()
      outputs = model(imgs)
      loss = loss_fn(outputs, labels)

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

      loss_train += loss.item()
    if epoch == 0 or epoch % 10 == 9:
      print('{} Epoch: {} loss: {}'.format(datetime.datetime.now(), epoch, loss_train/len(trainloader)))

In [ ]:
training_loop(100, model1, loss_fn, optimizer, trainloader)

2024-11-07 03:11:49.207436 Epoch: 0 loss: 0.9491145648854844
2024-11-07 03:11:54.540481 Epoch: 9 loss: 0.6212496609129804
2024-11-07 03:12:00.124553 Epoch: 19 loss: 0.501401881207811
2024-11-07 03:12:06.057566 Epoch: 29 loss: 0.43391628182948905
2024-11-07 03:12:11.481148 Epoch: 39 loss: 0.379409463101245
2024-11-07 03:12:17.124555 Epoch: 49 loss: 0.3356512964405912
2024-11-07 03:12:22.735857 Epoch: 59 loss: 0.2960002790106104
2024-11-07 03:12:28.226680 Epoch: 69 loss: 0.2573024003746662
2024-11-07 03:12:33.931033 Epoch: 79 loss: 0.21792398890916337
2024-11-07 03:12:39.361523 Epoch: 89 loss: 0.18284788721419395
2024-11-07 03:12:45.113924 Epoch: 99 loss: 0.14996711865384527


#4. Evaluation <a id="evaluation"></a>
Evaluate the model on the test dataset.

In [ ]:
def validate(model, trainloader, valLoader):
  for name, loader in [('train', trainloader), ('val', valLoader)]:
    correct = 0
    total = 0
    with torch.no_grad():
      for imgs, labels in loader:
        imgs = imgs.to(device = device)
        labels = labels.to(device = device)
        outputs = model(imgs)
        _, predicted = torch.max(outputs, dim = 1)
        total += labels.shape[0]
        correct += int((predicted == labels).sum())
    print('{}: {}'.format(name, correct/total))

In [ ]:
validate(model1, trainloader, valLoader)

train: 0.9569333333333333
val: 0.8173333333333334


In [ ]:
torch.save(model1.state_dict(), path + 'cnn1.pt')

In [ ]:
loaded = Net().to(device = device)
loaded.load_state_dict(torch.load(path + 'cnn1.pt', map_location = device))

<ipython-input-63-3ae714e706e7>:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  loaded.load_state_dict(torch.load(path + 'cnn1.pt', map_location = device))


<All keys matched successfully>